In [1]:
import pandas as pd

In [1]:
%pip install flask 

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/7e/d4/7ebdbd03970677812aac39c869717059dbb71a4cfc033ca6e5221787892c/click-8.1.8-py3-none-any.whl (98 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
DataStep7 = pd.read_feather('../Data/V2-DataStep7.feather')
X = DataStep7.drop(columns=['Cluster'])
y = DataStep7['Cluster']

In [4]:
X

,Basal_metabolic_rate,Standing_height,Sex,Testosterone,Weight,Waist_circumference,FVC,FEV1,Creatinine,Urate
0,5088.0,155.0,Female,0.737,65.6,77.0,2.28,1.63,55.7,351.7
1,6309.0,168.0,Female,0.380,81.0,84.0,3.40,2.77,67.1,286.6
2,5042.0,150.0,Female,0.852,57.7,70.0,2.97,2.14,57.2,163.5
3,5594.0,163.0,Female,1.564,51.0,71.0,4.15,2.97,63.9,170.0
4,5661.0,167.0,Female,1.093,71.9,88.0,3.05,2.41,51.4,270.4
...,...,...,...,...,...,...,...,...,...,...
35538,5724.0,159.0,Female,1.098,71.0,81.0,3.84,3.03,66.4,415.9
35539,6180.0,173.0,Female,1.415,86.9,84.0,3.54,2.55,66.9,267.2
35540,4845.0,161.0,Female,1.086,53.2,68.0,3.40,2.60,55.0,146.2
35541,6343.0,172.0,Female,1.094,91.8,86.0,3.96,2.76,65.1,235.5


In [3]:
from flask import Flask, request, jsonify
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import VarianceThreshold
import joblib

# 初始化 Flask 应用
app = Flask(__name__)
model_path = 'D:/UKBiobank/ML_Stroke/Models/V2-logistic_best_model.pkl'
with open(model_path, 'rb') as f:
    model = joblib.load(f)


# 定义数值和分类特征列
numeric_cols = ['Basal_metabolic_rate', 'Standing_height', 'Testosterone', 'Weight', 'Waist_circumference', 'FVC', 'FEV1', 'Creatinine', 'Urate']
categorical_cols = ['Sex']

# 创建预处理管道
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),           # 标准化数值列
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)  # 独热编码分类列
    ])

# 创建完整管道
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('variance_threshold', VarianceThreshold())           # 移除零方差特征
])

# 定义预测路由
@app.route('/predict', methods=['POST'])
def predict():
    # 从请求中获取 JSON 数据
    data = request.json

    # 将单条数据转换为 DataFrame
    input_data = pd.DataFrame([data])

    # 使用管道进行预处理
    processed_data = pipeline.transform(input_data)

    # 使用模型进行预测
    prediction_probability = model.predict_proba(processed_data)[0].tolist()

    # 返回预测结果
    return jsonify({
        'prediction_probability': prediction_probability
    })

# 运行应用
if __name__ == '__main__':
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

e:\Users\admin\miniconda3\envs\GTM\lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
from flask import Flask, request, jsonify
import pickle

app = Flask(__name__)

# 加载模型
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

@app.route('/predict', methods=['POST'])
def predict():
    data = request.json
    prediction = model.predict_proba([data['features']])
    return jsonify({'prediction_probability': prediction.tolist()})

if __name__ == '__main__':
    app.run(debug=True)



In [5]:
import requests

url = 'http://127.0.0.1:5000/predict'
data = {'features': [1, 2, 3, 4]}
response = requests.post(url, json=data)
print(response.json())

{'data_preview': [{'Basal_metabolic_rate': 5088.0, 'Cluster': '2', 'Creatinine': 55.7, 'FEV1': 1.63, 'FVC': 2.28, 'Sex': 'Female', 'Standing_height': 155.0, 'Testosterone': 0.737, 'Urate': 351.7, 'Waist_circumference': 77.0, 'Weight': 65.6}, {'Basal_metabolic_rate': 6309.0, 'Cluster': '2', 'Creatinine': 67.1, 'FEV1': 2.77, 'FVC': 3.4, 'Sex': 'Female', 'Standing_height': 168.0, 'Testosterone': 0.38, 'Urate': 286.6, 'Waist_circumference': 84.0, 'Weight': 81.0}, {'Basal_metabolic_rate': 5042.0, 'Cluster': '2', 'Creatinine': 57.2, 'FEV1': 2.14, 'FVC': 2.97, 'Sex': 'Female', 'Standing_height': 150.0, 'Testosterone': 0.852, 'Urate': 163.5, 'Waist_circumference': 70.0, 'Weight': 57.7}, {'Basal_metabolic_rate': 5594.0, 'Cluster': '2', 'Creatinine': 63.9, 'FEV1': 2.97, 'FVC': 4.150000000000001, 'Sex': 'Female', 'Standing_height': 163.0, 'Testosterone': 1.564, 'Urate': 170.0, 'Waist_circumference': 71.0, 'Weight': 51.0}, {'Basal_metabolic_rate': 5661.0, 'Cluster': '2', 'Creatinine': 51.4, 'FEV1'